### **Import Libraries**

In [ ]:
from pathlib import Path

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql import types as T

from pyspark.ml import Pipeline
from pyspark.ml.feature import (
    StringIndexer,
    OneHotEncoder,
    Imputer,
    VectorAssembler,
    StandardScaler
)

### **Initialize Spark Session**

In [ ]:
spark = (
    SparkSession.builder
    .appName("EEG_Data_Preprocessing")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")

### **Define Paths**

In [ ]:
project_root = Path("..").resolve()

input_dir = project_root / "data" / "processed" / "feature_engineered" / "final_modeling_table"
demographic_path = project_root / "data" / "demographic.csv"

output_dir = project_root / "data" / "processed" / "preprocessed_final_modeling_table"
pipeline_dir = project_root / "data" / "processed" / "preprocessing_pipeline"

output_dir.mkdir(parents=True, exist_ok=True)
pipeline_dir.mkdir(parents=True, exist_ok=True)

### **Load the Engineered Modeling Table**

In [ ]:
df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(str(input_dir))
)

print(f"Number of columns in modeling table: {len(df.columns)}")
df.select("subject", "trial", "condition").show(5, truncate=False)

### **Load the Demographic Table**

In [ ]:
demographic_df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(str(demographic_path))
)

demographic_df.show(5, truncate=False)

### **Remove Any Duplicate Rows**

In [ ]:
df = df.dropDuplicates(["subject", "trial", "condition"])

### **Clean and Prepare the Target Labels**

In [ ]:
print(demographic_df.columns)

In [ ]:
demographic_df = demographic_df.dropDuplicates(["subject"])

demographic_df = demographic_df.withColumnRenamed(" group", "label")

demographic_df = demographic_df.select("subject", "label")

### **Join the Target Labels to the Trial-Level Table**

In [ ]:
df = df.join(
    demographic_df,
    on="subject",
    how="left"
)

### **Validate the Target Join**

In [ ]:
df.select(
    "subject",
    "trial",
    "condition",
    "label"
).show(10, truncate=False)

### **Identify Missing Values per Column**

In [ ]:
missing_summary = df.select([
    F.sum(F.col(c).isNull().cast("int")).alias(c)
    for c in df.columns
])

missing_summary.show(truncate=False)

### **Define Identifier, Target, Categorical, and Numeric Columns**

In [ ]:
id_cols = ["subject", "trial"]
target_col = "label"
categorical_cols = ["condition"]

excluded_cols = set(id_cols + [target_col] + categorical_cols)

numeric_cols = [
    field.name
    for field in df.schema.fields
    if field.name not in excluded_cols
    and isinstance(
        field.dataType,
        (
            T.IntegerType,
            T.LongType,
            T.FloatType,
            T.DoubleType,
            T.ShortType,
            T.DecimalType
        )
    )
]

print(f"Categorical columns: {categorical_cols}")
print(f"Numeric columns: {len(numeric_cols)}")

### **Convert Categorical Columns to String Type**

In [ ]:
for c in categorical_cols:
    df = df.withColumn(c, F.col(c).cast("string"))

### **Define the Categorical Encoding Stage**

In [ ]:
indexer_stages = [
    StringIndexer(
        inputCol=col_name,
        outputCol=f"{col_name}_indexed",
        handleInvalid="keep"
    )
    for col_name in categorical_cols
]

### **Define the One-Hot Encoding Stage**

In [ ]:
encoded_input_cols = [f"{c}_indexed" for c in categorical_cols]
encoded_output_cols = [f"{c}_ohe" for c in categorical_cols]

encoder_stage = OneHotEncoder(
    inputCols=encoded_input_cols,
    outputCols=encoded_output_cols,
    handleInvalid="keep"
)

### **Define the Feature Assembly Stage**

In [ ]:
feature_input_cols = numeric_cols + encoded_output_cols

assembler_stage = VectorAssembler(
    inputCols=feature_input_cols,
    outputCol="features_raw",
    handleInvalid="skip"
)

### **Define the Feature Scaling Stage**

In [ ]:
scaler_stage = StandardScaler(
    inputCol="features_raw",
    outputCol="features",
    withMean=False,
    withStd=True
)

### **Build the Complete Preprocessing Pipeline**

In [ ]:
preprocessing_pipeline = Pipeline(
    stages=indexer_stages + [
        encoder_stage,
        assembler_stage,
        scaler_stage
    ]
)

### **Fit the Preprocessing Pipeline and Transform the Dataset**

In [ ]:
preprocessing_model = preprocessing_pipeline.fit(df)
preprocessed_df = preprocessing_model.transform(df)

### **Review the Preprocessed Output**

In [ ]:
preprocessed_df.select(
    "subject",
    "trial",
    "condition",
    "label",
    "features_raw",
    "features"
).show(5, truncate=False)

### **Build the Final Preprocessed Modeling Table**

In [ ]:
final_preprocessed_df = preprocessed_df.select(
    "subject",
    "trial",
    "condition",
    "label",
    "features"
)

In [ ]:
final_preprocessed_df.select(
    "subject",
    "trial",
    "condition",
    "label"
).show(10, truncate=False)

### **Save the Final Preprocessed Dataset**

In [ ]:
(
    final_preprocessed_df
    .write
    .mode("overwrite")
    .parquet(str(output_dir))
)

### **Save the Fitted Preprocessing Pipeline**

In [ ]:
preprocessing_model.write().overwrite().save(str(pipeline_dir))